In [1]:
import os
import sys
#切换到当前包含merkle.py的文件夹
os.chdir(r"C:\Users\DELL\Desktop\SageMathLab\FRI")

from merkle import MerkleTree, hash 

F=GF(97)
R.<x> = F[]

OMEGA=F.multiplicative_generator() #生成元

P=R.random_element(degree=9)
print("P: ",P) #生成一个degree=9的多项式
P_degree=P.degree()
BLOWUP_FACTOR=4


47525c1e09ea1731d1a8e187f640527cc8ce4f444ddcb398b55961dbb5403222
47525c1e09ea1731d1a8e187f640527cc8ce4f444ddcb398b55961dbb5403222
P:  50*x^9 + 61*x^8 + 5*x^7 + 29*x^6 + 59*x^5 + x^4 + 56*x^3 + 9*x^2 + 73*x + 28


/cygdrive/c/Users/DELL/Desktop/SageMathLab/FRI/merkle.py:46: DeprecationWarning: invalid escape sequence \ 
  '''


In [10]:
from __future__ import annotations

#将系数分成偶数和奇数
def split_polynomial(poly:R)->tuple[R,R]:
    coeffs=poly.coefficients(sparse=False)
    deg=poly.degree()
    Pe=sum(coeffs[i]*x^(i//2) for i in range(0,deg+1,2)) #遍历偶次项系数0,2,4,6,8
    Po=sum(coeffs[i]*x^((i-1)//2) for i in range(1,deg+1,2)) #遍历奇次项系数1,3,5,7,9
    assert(Pe(x^2)+x*Po(x^2)==poly)
    return Pe, Po

#用随机值alpha折叠(commit)
def fold_polynomial(poly:R,alpha:F)->R: #返回一个多项式
    Pe,Po=split_polynomial(poly)
    Pf=Pe+alpha*Po

    # [新增输出] 展示折叠的内部细节
    print(f"    ├─ 拆分 偶数项 Pe(x): {Pe}")
    print(f"    ├─ 拆分 奇数项 Po(x): {Po}")
    print(f"    └─ 混合 (Pe + {alpha} * Po): {Pf}") 
    return Pf

#在domain上对多项式进行求值 (LDE低度数扩展)
def evaluate_polynomial(poly:R,blowup:int,omega:F)->list[F]: #blowup是膨胀因子
    domain=(P_degree+1)*blowup #定义了求值点的总数
    #如果原多项式的阶是d,那么最少需要d+1个点就能确定这个多项式
    #但是为了安全性,通常会在比最小点集更大的范围内求值
    
    evals=[poly(omega^i) for i in range(domain)]
    # [新增输出] 展示求值域大小
    print(f"    ├─ 定义域大小: {domain} 个点 (Blowup Factor: {blowup})")

    return evals
    #[P(w^0),P(w^1),...P(w^n-1)]

#commitment: 生成merkle tree
def commit_polynomial(poly:R,blowup:int,omega:F)->MerkleTree:
    evaluations=evaluate_polynomial(poly,BLOWUP_FACTOR,omega) #LDE
    
    #将eval转换为int，并指定字节长度(32字节足以容纳GF(97))
    leaves=[hash(int(eval).to_bytes(32,'big')) for eval in evaluations] 
    return MerkleTree(leaves) 

#一轮 FRI commitment:
#- 计算merkle tree
#- 折叠多项式
def round(poly:R,omega:F)->tuple[MerkleTree,R,F]:
    tree=commit_polynomial(poly,BLOWUP_FACTOR,omega)
    print(f"    ├─ 生成 Merkle Tree，根哈希: 0x{tree.root().hex()}...")

    alpha=F.random_element()
    print(f"  >> 验证者给出随机挑战数 alpha: {alpha}")

    folded=fold_polynomial(poly,alpha)
    return (tree,folded,alpha)

#对多项式进行折叠和合并，直到我们得到一个常数（0次多项式）:
def commit(poly:R)->tuple[list[MerkleTree],list[Any],list[F]]:
    print("\n========== FRI 协议开始: COMMIT 阶段 ==========")
    trees=[]
    polys=[]
    alphas=[]
    omega=OMEGA

    polys.append(poly)
    while poly.degree()>0: #只要多项式度数大于0
        (tree,folded,alpha)=round(poly,omega)
        trees.append(tree)
        alphas.append(alpha)
        polys.append(folded)

        poly=folded
        omega=pow(omega,2) 
        
        #因为28^32=1(mod 97)，有32个叶子
        #所以(28^2)^16=1(mod 97)，有16个叶子

    print(f"\n[COMMIT 阶段] --- 最终轮 ---")
    print(f"  多项式已完全折叠为常数 (度数 0): {poly}")
    print("===============================================")
    return (trees,polys,alphas)


#在索引z处打开承诺
#意味着:
# - 计算 f(z) 和 f(-z)
# - 生成 merkle tree 证明
def query(z:F,polys:list[R],trees:list[MerkleTree])->list[tuple[F,F,list[bytes]]]:
    print(f"\n========== FRI 协议: QUERY 阶段 ==========")
    print(f"验证者要求抽查初始索引: z = {z}")

    omega=OMEGA^z #随机选取一个点z
    queries=[]
    for i in range(len(trees)): #遍历树的layer
        print(f"\n[QUERY 阶段] --- 第 {i} 棵树 ---")
        f_z=polys[i](omega) #带入omega值计算多项式poly的值
        f_z_minus=polys[i](-omega)
        print(f"  计算点: omega = {omega}, -omega = {-omega}")
        print(f"  多项式求值: f(z) = {f_z}, f(-z) = {f_z_minus}")

        merkle_proof=trees[i].proof(z) #证明这个点确实存在于Merkle tree中
        print(f"  生成叶子索引 {z} 的 Merkle 证明 (路径长度: {len(merkle_proof)})")

        queries.append((f_z,f_z_minus,merkle_proof))
        omega=pow(omega,2)
    return queries

#验证所有树层是否匹配
def verify(z:F,queries:list[tuple[F,F,list[bytes]]],commitments:list[bytes],alphas:list[F]):
    print(f"\n========== FRI 协议: VERIFY 阶段 ==========")

    omega=OMEGA^z
    previous_round=None
    for i in range(len(queries)):
        print(f"\n[VERIFY 阶段] --- 验证第 {i} 层 ---")

        f_z=queries[i][0]
        f_z_minus=queries[i][1]
        merkle_proof=list(queries[i][2])

        root=MerkleTree.compute_tree_from_proof(int(f_z).to_bytes(32,'big'),merkle_proof) #通过一个随机的叶节点来求根节点

        
        print(f"  收到 f(z) = {f_z}, f(-z) = {f_z_minus}")

        if root == commitments[i]:
            print(f"  [通过] Merkle Root 匹配: 0x{root.hex()}...")
        else:
            raise AssertionError(f"Merkle 验证失败在第 {i} 层！")
        
        if previous_round is not None:
            if f_z == previous_round:
                print(f"  [通过] 跨层一致性验证: 本层 f(z) {f_z} == 上层推导预期值 {previous_round}")
            else:
                raise AssertionError(f"跨层验证失败！预期 {previous_round}，实际 {f_z}")
            
        verif=(((alphas[i]+omega)/(2*omega))*f_z)+(((alphas[i]-omega)/(2*(-omega)))*f_z_minus)
        print(f"  >> 根据折叠公式计算下一层预期值为: {verif}")
        previous_round=verif
        omega=pow(omega,2)

    # 最后一轮检查常数
    print(f"\n[VERIFY 阶段] --- 检查最终常数 ---")
    if previous_round == commitments[-1]:
        print(f"  [通过] 最终折叠值 {previous_round} == 证明者承诺的常数 {commitments[-1]}")
        print("\n🎉🎉🎉 FRI 协议验证成功！原始多项式确实是一个低度数多项式！ 🎉🎉🎉\n")
    else:
        raise AssertionError("最终常数验证失败！")


In [12]:
(trees,polys,alphas)=commit(P)
commitments=[tree.root() for tree in trees]+[polys[-1]] #root 哈希+最后的常数
# 遍历 commitments，如果是字节就转成 16 进制，如果是数字就直接显示
readable_commitments = [
    c.hex() if isinstance(c, bytes) else c #检查是不是字节串如果是则转换成16进制字符串
    for c in commitments
 ]

for i, c in enumerate(readable_commitments):
    if i < len(readable_commitments) - 1:
        print(f"第 {i} 层 Merkle Root: 0x{c}")
    else:
        print(f"最终折叠常数: {c}")
print("=========================================================================")
z=12
assert(z<=BLOWUP_FACTOR*P_degree)

queries=query(z,polys,trees)
verify(z,queries,commitments,alphas)


========== FRI 协议开始: COMMIT 阶段 ==========
    ├─ 定义域大小: 40 个点 (Blowup Factor: 4)
    ├─ 生成 Merkle Tree，根哈希: 0xab968936a036fb48177807df3ce1a2ec148663ca85e2cb2d0297cc03756dafda...
  >> 验证者给出随机挑战数 alpha: 57
    ├─ 拆分 偶数项 Pe(x): 61*x^4 + 29*x^3 + x^2 + 9*x + 28
    ├─ 拆分 奇数项 Po(x): 50*x^4 + 5*x^3 + 59*x^2 + 56*x + 73
    └─ 混合 (Pe + 57 * Po): x^4 + 23*x^3 + 66*x^2 + 18
    ├─ 定义域大小: 40 个点 (Blowup Factor: 4)
    ├─ 生成 Merkle Tree，根哈希: 0xe86c738aec2286f7f780301ac2fb26fc3f40f65cb003b5a68e44192124ca9df7...
  >> 验证者给出随机挑战数 alpha: 42
    ├─ 拆分 偶数项 Pe(x): x^2 + 66*x + 18
    ├─ 拆分 奇数项 Po(x): 23*x
    └─ 混合 (Pe + 42 * Po): x^2 + 62*x + 18
    ├─ 定义域大小: 40 个点 (Blowup Factor: 4)
    ├─ 生成 Merkle Tree，根哈希: 0x65d4ffb146d25da7638ff05f9321a7fea8f36d88fa9e773cf7e5f02219ff8f6a...
  >> 验证者给出随机挑战数 alpha: 81
    ├─ 拆分 偶数项 Pe(x): x + 18
    ├─ 拆分 奇数项 Po(x): 62
    └─ 混合 (Pe + 81 * Po): x + 93
    ├─ 定义域大小: 40 个点 (Blowup Factor: 4)
    ├─ 生成 Merkle Tree，根哈希: 0xd399421ad43aa485fe3d546579d8409b8d37b159400f3edc2